# Extra Credit: Spark vs Ray Framework Comparison
Beehive Sounds Dataset — SBCM

**Task:** Compute statistics (mean, std, percentiles) on the CSV dataset using both Spark and Ray, then compare performance.

In [ ]:
import sys
sys.path.append('/home/stiwari6/.local/lib/python3.11/site-packages')

import kagglehub
import os
import time
import tracemalloc

os.environ["KAGGLE_USERNAME"] = "snigdhatiwarisd"
os.environ["KAGGLE_KEY"] = "KGAT_b5f7b91d08ee5010cc63d7ee3b3e3170"

path = kagglehub.dataset_download("annajyang/beehive-sounds")
csv_path = os.path.join(path, "all_data_updated.csv")
print("csv_path:", csv_path)

In [ ]:
CONTINOUS_COLS = [
    "hive temp", "hive humidity", "hive pressure",
    "weather temp", "weather humidity", "weather pressure",
    "wind speed", "gust speed", "cloud coverage",
    "rain", "lat", "long", "frames", "time"
]

## Spark Implementation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("BeehiveStats_Spark") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

df = spark.read.csv(csv_path, header=True, inferSchema=True)
df.cache()
df.count()  # materialize cache
print(f"loaded {df.count()} rows")

In [ ]:
def run_spark_stats():
    # mean, std, min, max, count
    df.select([F.col(f"`{c}`") for c in CONTINOUS_COLS]).describe().collect()

    # percentiles (Q1, median, Q3)
    quantile_aggs = []
    for c in CONTINOUS_COLS:
        quantile_aggs += [
            F.expr(f"percentile_approx(`{c}`, 0.25)").alias(f"{c}_Q1"),
            F.expr(f"percentile_approx(`{c}`, 0.50)").alias(f"{c}_median"),
            F.expr(f"percentile_approx(`{c}`, 0.75)").alias(f"{c}_Q3"),
        ]
    df.agg(*quantile_aggs).collect()

spark_times = []
for i in range(3):
    start = time.time()
    run_spark_stats()
    elapsed = time.time() - start
    spark_times.append(elapsed)
    print(f"  run {i+1}: {elapsed:.2f}s")

spark_avg = sum(spark_times) / len(spark_times)
print(f"\nSpark average: {spark_avg:.2f}s")

In [ ]:
# peak memory for spark
tracemalloc.start()
run_spark_stats()
_, spark_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
spark_peak_mb = spark_peak_bytes / 1024 / 1024
print(f"Spark peak memory: {spark_peak_mb:.1f} MB")

In [ ]:
# show the actual results as formatted tables
import pandas as pd

# ── descriptive stats ────────────────────────────────────────
desc_pd = (
    df.select([F.col(f"`{c}`") for c in CONTINOUS_COLS])
      .describe()
      .toPandas()
      .set_index("summary")
      .T
      .rename_axis("column")
      .reset_index()
)
for col in desc_pd.columns[1:]:
    desc_pd[col] = pd.to_numeric(desc_pd[col], errors="coerce").round(3)

print("=== Spark Descriptive Statistics ===")
print(desc_pd.to_string(index=False))

# ── percentiles ───────────────────────────────────────────────
quantile_aggs = []
for c in CONTINOUS_COLS:
    quantile_aggs += [
        F.expr(f"percentile_approx(`{c}`, 0.25)").alias(f"{c}_Q1"),
        F.expr(f"percentile_approx(`{c}`, 0.50)").alias(f"{c}_median"),
        F.expr(f"percentile_approx(`{c}`, 0.75)").alias(f"{c}_Q3"),
    ]

q_pd = df.agg(*quantile_aggs).toPandas()

rows = []
for c in CONTINOUS_COLS:
    rows.append({
        "column":  c,
        "Q1":      round(q_pd[f"{c}_Q1"].iloc[0], 3),
        "median":  round(q_pd[f"{c}_median"].iloc[0], 3),
        "Q3":      round(q_pd[f"{c}_Q3"].iloc[0], 3),
    })
q_df = pd.DataFrame(rows)

print("\n=== Spark Percentiles ===")
print(q_df.to_string(index=False))

## Ray Implementation

In [ ]:
import ray
import ray.data
import numpy as np

ray.init(ignore_reinit_error=True)
print(ray.available_resources())

In [ ]:
def run_ray_stats():
    ds = ray.data.read_csv(csv_path)

    # mean, std, min, max per column
    stats = {}
    for col in CONTINOUS_COLS:
        stats[col] = {
            "mean":  ds.mean(col),
            "std":   ds.std(col),
            "min":   ds.min(col),
            "max":   ds.max(col),
            "count": ds.count(),
        }

    # percentiles via pandas conversion on the numeric subset
    pdf = ds.select_columns(CONTINOUS_COLS).to_pandas()
    quantiles = pdf.quantile([0.25, 0.50, 0.75])

    return stats, quantiles

ray_times = []
for i in range(3):
    start = time.time()
    run_ray_stats()
    elapsed = time.time() - start
    ray_times.append(elapsed)
    print(f"  run {i+1}: {elapsed:.2f}s")

ray_avg = sum(ray_times) / len(ray_times)
print(f"\nRay average: {ray_avg:.2f}s")

In [ ]:
# peak memory for ray
tracemalloc.start()
run_ray_stats()
_, ray_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
ray_peak_mb = ray_peak_bytes / 1024 / 1024
print(f"Ray peak memory: {ray_peak_mb:.1f} MB")

In [ ]:
# show actual results
ray_stats, ray_quantiles = run_ray_stats()

print("=== Ray Descriptive Statistics ===")
import pandas as pd
rows = []
for col, s in ray_stats.items():
    rows.append({"column": col, "count": s["count"], "mean": round(s["mean"],4),
                 "std": round(s["std"],4), "min": round(s["min"],4), "max": round(s["max"],4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== Ray Percentiles ===")
print(ray_quantiles.to_string())

## Performance Comparison

In [ ]:
import pandas as pd

spark_loc = """df.select(...).describe().collect()
df.agg(*quantile_aggs).collect()""".strip().count('\n') + 2 + len(CONTINOUS_COLS)*3 + 5

ray_loc = """ds = ray.data.read_csv(csv_path)
for col in CONTINOUS_COLS: ds.mean/std/min/max/count
pdf = ds.select_columns(...).to_pandas()
quantiles = pdf.quantile([0.25,0.50,0.75])""".strip().count('\n') + 4

comparison = pd.DataFrame({
    "Metric":         ["Avg execution time (3 runs)", "Lines of code", "Peak memory (MB)"],
    "Spark":          [f"{spark_avg:.2f}s", "~15", f"{spark_peak_mb:.1f}"],
    "Ray":            [f"{ray_avg:.2f}s",   "~20", f"{ray_peak_mb:.1f}"],
})
print(comparison.to_string(index=False))

## Analysis

**Which framework was faster? By how much?**
Spark was faster for this task. The beehive CSV sits on Expanse's Lustre filesystem, and `describe()` + `percentile_approx` are single-pass aggregations that Spark handles efficiently in one call. Ray reads the CSV sequentially before distributing the work, so the overhead ends up outweighing any parallelism benefit at this dataset size.

**Which was easier to implement? Why?**
Spark was easier to implement. `df.describe()` and `percentile_approx` produce a clean summary in one call. Ray Data doesn't have a built-in percentile aggregator, so we had to convert to pandas to compute quantiles, which added an extra step and made the code more complicated.

**For your specific use case, which would you choose?**
We would choose Spark for tabular statistics and preprocessing. Ray would be the better choice if the pipeline included the audio feature extraction UDF at scale or model training with PyTorch, since Ray's integrations (Ray Train, Ray Tune) give it a clear advantage over Spark's MLlib for those kinds of tasks.